# Task 3: Interpret the Embedding Space

This notebook addresses two questions in sequence:

1. **Is disease state linearly encoded in GeneFormer embeddings?**
   We fit one LDA per cell type on the baseline embeddings. LDA finds the single
   linear direction in the 512-dimensional space that maximally separates sALS from
   PN, the disease axis. 5-fold cross-validation quantifies how well this axis
   discriminates the two groups.

2. **Do single-gene perturbations move sALS cells toward PN along this axis?**
   For each of the 24 perturbations (12 genes × knockup/knockdown) we measure two
   complementary metrics: LDA score shift (global, linear) and KNN neighborhood
   shift (local, non-linear). Each metric is evaluated against three reliability
   criteria (0-3 each), producing per-metric scores that are passed to Task 4.

In [ ]:
# Imports
%load_ext autoreload
%autoreload 2

import pathlib
import scipy.sparse as sp

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import NearestNeighbors

from src.data import load_data, load_embeddings
from src.constants import RELEVANT_CELLTYPES, SALS_GENES

RESULTS_DIR = pathlib.Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
# Load data, embeddings, and cell-group masks
adata = load_data()
sub_index = pd.read_csv("../data/adata_sub_index.csv").squeeze()
adata_sub = adata[sub_index].copy()
obs = adata_sub.obs.reset_index(drop=True)
print(f"Loaded subset: {adata_sub.n_obs:,} cells")

embeddings = load_embeddings()
print(f"Available embeddings: {list(embeddings.keys())}")

sals_mask = (obs["Group"] == "SALS").values
pn_mask = (obs["Group"] == "PN").values
pert_keys = [k for k in embeddings if k != "baseline"]
emb_baseline = embeddings["baseline"]

## Plot 0: Gene expression reference

Mean expression of each target gene per cell type and disease group. Darker blue
means higher expression. This is the reference for all downstream direction
checks: a gene more highly expressed in PN than sALS should, under a simple
rank-shift model, show a positive LDA/KNN shift after knockup and a negative
shift after knockdown. Deviations from this expectation are flagged in the
reliability table.

In [ ]:
# Expression heatmap: mean per (gene, celltype, group)
X_sub = adata_sub.X.toarray() if sp.issparse(adata_sub.X) else adata_sub.X

expr_rows = []
for gene in SALS_GENES:
    if gene not in adata_sub.var_names:
        continue
    g_idx = adata_sub.var_names.get_loc(gene)
    for ct in RELEVANT_CELLTYPES:
        ct_mask = (obs["Cellstates_LVL3"] == ct).values
        for group, mask in [("sALS", sals_mask), ("PN", pn_mask)]:
            expr_rows.append({
                "gene": gene,
                "celltype": ct,
                "group": group,
                "mean_expr": X_sub[ct_mask & mask, g_idx].mean(),
            })

expr_df = pd.DataFrame(expr_rows)
expr_pivot = expr_df.pivot_table(
    index="gene", columns=["celltype", "group"], values="mean_expr"
)

col_order = [(ct, grp) for ct in RELEVANT_CELLTYPES for grp in ["sALS", "PN"]]
expr_pivot = expr_pivot[col_order]

from matplotlib.colors import LogNorm
fig, ax = plt.subplots(figsize=(len(col_order) * 1.1, len(SALS_GENES) * 0.55))
im = ax.imshow(
    expr_pivot.values,
    aspect="auto",
    cmap="Blues",
    norm=LogNorm(vmin=max(expr_pivot.values.min(), 1e-3), vmax=expr_pivot.values.max())
)
plt.colorbar(im, ax=ax, label="Mean expression (log scale)")

ax.set_xticks(range(len(col_order)))
ax.set_xticklabels([grp for _, grp in col_order], fontsize=8)
ax.set_yticks(range(len(expr_pivot)))
ax.set_yticklabels(expr_pivot.index)

for i, ct in enumerate(RELEVANT_CELLTYPES):
    x_frac = (i * 2 + 1) / len(col_order)
    ax.text(
        x_frac, -0.07, ct.replace("_", " "),
        ha="center", va="top", fontsize=7,
        transform=ax.transAxes
    )
    if i > 0:
        ax.axvline(i * 2 - 0.5, color="black", lw=1.5)

ax.set_title("Mean expression per gene, cell type, and disease group")
plt.tight_layout()
plt.savefig("../results/expression_reference.png", dpi=150, bbox_inches="tight")
plt.show()

## Part 1: Is disease state linearly encoded?

LDA is fit on 512-dimensional GeneFormer embeddings with 5-fold cross-validation.
One model per cell type prevents cell-type variation from leaking into the disease
axis. The sign of each model is corrected so that **higher LDA score always means
more PN-like**, enabling consistent interpretation across all downstream plots.

In [ ]:
# Fit one LDA per cell type; correct sign so higher score = more PN-like
y = (obs["Group"] == "SALS").astype(int).values  # sALS=1, PN=0
ldas = {}
lda_signs = {}
cv_scores = {}
baseline_scores = np.full(len(obs), np.nan)

for ct in RELEVANT_CELLTYPES:
    ct_mask = (obs["Cellstates_LVL3"] == ct).values
    X_ct = emb_baseline[ct_mask]
    y_ct = y[ct_mask]

    lda_ct = LinearDiscriminantAnalysis()
    cv = cross_val_score(lda_ct, X_ct, y_ct, cv=5, scoring="accuracy")
    cv_scores[ct] = cv
    print(f"{ct}: CV accuracy {cv.mean():.3f} ± {cv.std():.3f}")

    lda_ct.fit(X_ct, y_ct)
    raw = lda_ct.decision_function(X_ct)
    sign = -1 if raw[y_ct == 0].mean() < raw[y_ct == 1].mean() else 1
    lda_signs[ct] = sign
    baseline_scores[ct_mask] = sign * raw
    ldas[ct] = lda_ct

    sals_ct = sals_mask & ct_mask
    pn_ct = pn_mask & ct_mask
    print(f"  sALS mean: {baseline_scores[sals_ct].mean():.3f} | PN mean: {baseline_scores[pn_ct].mean():.3f}")

## Plot 1: Baseline LDA score distributions

Violin plots of per-cell LDA scores for sALS (red) and PN (blue) in the baseline
embedding. The gap between the two violins is the distance that perturbations
would need to close to achieve rescue. CV accuracy confirms the disease axis is
real, not an artifact of overfitting.

In [ ]:
# Violin plot of baseline LDA scores per cell type
fig, axes = plt.subplots(
    1, len(RELEVANT_CELLTYPES),
    figsize=(5 * len(RELEVANT_CELLTYPES), 5),
    sharey=True
)
if len(RELEVANT_CELLTYPES) == 1:
    axes = [axes]

for ax, ct in zip(axes, RELEVANT_CELLTYPES):
    ct_mask = (obs["Cellstates_LVL3"] == ct).values
    for group, mask, color in [("PN", pn_mask, "#1f77b4"), ("sALS", sals_mask, "#d62728")]:
        scores = baseline_scores[ct_mask & mask]
        vp = ax.violinplot(
            [scores],
            positions=[0 if group == "PN" else 1],
            showmedians=True,
            showextrema=False
        )
        for pc in vp["bodies"]:
            pc.set_facecolor(color)
            pc.set_alpha(0.7)
        vp["cmedians"].set_color("black")

    acc = cv_scores[ct].mean()
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["PN", "sALS"])
    ax.set_title(f"{ct.replace('_', ' ')}\nCV acc = {acc:.2f}")

axes[0].set_ylabel("LDA score (higher = more PN-like)")
plt.suptitle("Baseline LDA score distributions by cell type (one LDA per cell type)")
plt.tight_layout()
plt.savefig("../results/lda_baseline_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## Part 2: LDA score shift after perturbation

The fixed baseline LDA is applied to each perturbation's embeddings. We record
only the change in mean LDA score for sALS cells, normalized by the baseline
sALS to PN gap: +1 means sALS cells fully reached the PN mean, -1 means an equal
displacement in the wrong direction.

In [ ]:
# Compute normalized LDA shift for sALS cells across all perturbations
lda_rows = []
for key in pert_keys:
    gene, direction = key.rsplit("_", 1)
    emb_pert = embeddings[key]
    for ct in RELEVANT_CELLTYPES:
        ct_mask = (obs["Cellstates_LVL3"] == ct).values
        sals_ct = sals_mask & ct_mask
        pn_ct = pn_mask & ct_mask
        pert_s = lda_signs[ct] * ldas[ct].decision_function(emb_pert[ct_mask])
        lda_gap = baseline_scores[pn_ct].mean() - baseline_scores[sals_ct].mean()
        local_sals = sals_mask[ct_mask]
        delta = (pert_s[local_sals].mean() - baseline_scores[sals_ct].mean()) / lda_gap
        lda_rows.append({"gene": gene, "direction": direction, "celltype": ct, "lda_shift": delta})

lda_df = pd.DataFrame(lda_rows)

## Plot 2: LDA shift heatmap

Each cell shows the normalized Δ LDA score for sALS cells after perturbation
(↑ = knockup, ↓ = knockdown) in one cell type. Red = movement toward PN (rescue);
blue = movement away. Genes are sorted top to bottom by mean shift across all
conditions.

In [ ]:
# Build and render LDA shift heatmap
directions = ["up", "down"]
col_tuples = [(ct, d) for ct in RELEVANT_CELLTYPES for d in directions]
n_cols = len(col_tuples)
n_ct = len(RELEVANT_CELLTYPES)

gene_order = (
    lda_df.groupby("gene")["lda_shift"].mean()
    .sort_values(ascending=False)
    .index.tolist()
)

heatmap_data = (
    lda_df.pivot(index="gene", columns=["celltype", "direction"], values="lda_shift")
    .loc[gene_order, col_tuples]
)

abs_max = heatmap_data.values.__abs__().max()
fig, ax = plt.subplots(figsize=(max(5, n_cols * 1.1), max(5, len(gene_order) * 0.6)))
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="RdBu_r", vmin=-abs_max, vmax=abs_max)
plt.colorbar(im, ax=ax, label="Δ LDA score (fraction of sALS→PN gap)")
ax.set_xticks(range(n_cols))
ax.set_xticklabels(["↑" if d == "up" else "↓" for _, d in col_tuples], fontsize=10)
ax.set_yticks(range(len(gene_order)))
ax.set_yticklabels(gene_order)
for i, ct in enumerate(RELEVANT_CELLTYPES):
    x_frac = (i * len(directions) + (len(directions) - 1) / 2 + 0.5) / n_cols
    ax.text(
        x_frac, -0.07, ct.replace("_", " "),
        ha="center", va="top", fontsize=8, transform=ax.transAxes
    )
for i in range(1, n_ct):
    ax.axvline(i * len(directions) - 0.5, color="black", lw=1.5)
ax.set_title(
    "LDA score shift: sALS cells moving toward PN\n"
    "(red = rescue, blue = away; normalized by sALS→PN gap)"
)
plt.tight_layout()
plt.savefig("../results/lda_shift_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

### Why these results are not trustworthy

Four observations suggest the heatmap reflects noise rather than a genuine rescue
signal. These map directly onto the reliability checks applied later:

**1. Effect sizes are negligible.** All Δ LDA values are ≤ 3% of the sALS→PN
gap. A biologically meaningful rescue would need to substantially close that gap.

**2. No consistency across cell types.** A genuine rescue gene should show the
same shift direction in all three cell types. Many genes flip sign between cell
types, failing the celltype agreement check.

**3. Knockup and knockdown often shift in the same direction.** A real gene-dose
effect would produce opposite-sign shifts for the two directions. Genes where both
KU and KD go the same way fail the KU/KD opposite check.

**4. Observed direction contradicts expression differences.** Most target genes
are more highly expressed in PN than sALS (see Plot 0). The expected knockup
direction is therefore rescue (positive shift). Genes where the observed direction
disagrees fail the KU exp=obs check. The reliability table below applies all
three checks independently to LDA and KNN, giving a 0-3 score for each metric.

## Part 3: KNN neighborhood shift

A local, non-linear complement to the LDA metric. A K=20 nearest-neighbor graph
is fit on the full baseline embedding, then for each perturbation we query where
perturbed sALS cells land in that fixed neighborhood structure. A positive delta
means a sALS cell moved into a more PN-rich local neighborhood, regardless of
whether that movement is along the LDA axis.

In [ ]:
# Fit baseline KNN graph; compute per-cell PN-neighbor fraction before and after each perturbation
nbrs = NearestNeighbors(n_neighbors=21, metric="cosine").fit(emb_baseline)
sals_idx = np.where(sals_mask)[0]
_, base_nn = nbrs.kneighbors(emb_baseline[sals_idx])
base_pn_frac = pn_mask[base_nn[:, 1:]].mean(axis=1)  # exclude self (col 0)

knn_rows = []
for key in pert_keys:
    gene, direction = key.rsplit("_", 1)
    _, pert_nn = nbrs.kneighbors(embeddings[key][sals_idx])
    pert_pn_frac = pn_mask[pert_nn[:, 1:]].mean(axis=1)
    delta = pert_pn_frac - base_pn_frac
    ct_labels = obs["Cellstates_LVL3"].values[sals_idx]
    for ct in RELEVANT_CELLTYPES:
        sel = ct_labels == ct
        knn_rows.append({
            "gene": gene, "direction": direction, "celltype": ct,
            "delta_pn_frac": delta[sel].mean(),
        })

knn_df = pd.DataFrame(knn_rows)

## Plot 3: KNN neighborhood shift heatmap

Same layout as the LDA heatmap. Red = sALS cells gained more PN neighbors after
perturbation; blue = lost PN neighbors. The same reliability issues are visible
as in the LDA heatmap: negligible effects, no cross-celltype consistency, KU and
KD often agree, and directions frequently contradict expression expectations.
Notably, the genes that appear strongest here often disagree with the LDA ranking,
indicating both metrics are dominated by noise rather than a biological signal.

In [ ]:
# Build and render KNN shift heatmap (same gene order as LDA heatmap)
knn_data = (
    knn_df.pivot(index="gene", columns=["celltype", "direction"], values="delta_pn_frac")
    .loc[gene_order, col_tuples]
)
abs_max_knn = knn_data.values.__abs__().max()

fig, ax = plt.subplots(figsize=(max(5, n_cols * 1.1), max(5, len(gene_order) * 0.6)))
im = ax.imshow(knn_data.values, aspect="auto", cmap="RdBu_r", vmin=-abs_max_knn, vmax=abs_max_knn)
plt.colorbar(im, ax=ax, label="Δ PN-neighbor fraction (K=20)")
ax.set_xticks(range(n_cols))
ax.set_xticklabels(["↑" if d == "up" else "↓" for _, d in col_tuples], fontsize=10)
ax.set_yticks(range(len(gene_order)))
ax.set_yticklabels(gene_order)
for i, ct in enumerate(RELEVANT_CELLTYPES):
    x_frac = (i * len(directions) + (len(directions) - 1) / 2 + 0.5) / n_cols
    ax.text(
        x_frac, -0.07, ct.replace("_", " "),
        ha="center", va="top", fontsize=8, transform=ax.transAxes
    )
for i in range(1, n_ct):
    ax.axvline(i * len(directions) - 0.5, color="black", lw=1.5)
ax.set_title(
    "KNN neighborhood shift: sALS cells gaining PN neighbors\n"
    "(red = toward PN, blue = away; K=20, cosine metric)"
)
plt.tight_layout()
plt.savefig("../results/knn_neighborhood.png", dpi=150, bbox_inches="tight")
plt.show()

## Reliability table

For each gene, three binary checks are applied independently to LDA and KNN,
giving a per-metric reliability score (0-3). True (✓) is always the good value;
False (✗) is a red flag.

| Check | What it tests |
|---|---|
| KU/KD opposite | KU and KD produce opposite-sign shifts (gene-dose coherence) |
| KU celltype agree | KU shift has the same sign in all three cell types |
| KU exp=obs | KU direction matches expectation from Plot 0 expression differences |

The per-metric scores (lda_reliability 0-3, knn_reliability 0-3) are passed to
Task 4 where each metric's SNR is weighted by its own reliability before the
composite is formed. A gene that passes no checks on a given metric contributes
nothing from that metric to the final score.

In [ ]:
# Build combined reliability table for both LDA and KNN
reliability_rows = []
for gene in SALS_GENES:
    if gene not in adata_sub.var_names:
        continue
    g_idx = adata_sub.var_names.get_loc(gene)
    expr_diff = X_sub[pn_mask, g_idx].mean() - X_sub[sals_mask, g_idx].mean()
    ku_expected_positive = expr_diff > 0

    lda_g = lda_df[lda_df["gene"] == gene]
    lda_ku_vals = lda_g[lda_g["direction"] == "up"]["lda_shift"].values
    lda_kd_vals = lda_g[lda_g["direction"] == "down"]["lda_shift"].values
    lda_ku = lda_ku_vals.mean()
    lda_kd = lda_kd_vals.mean()

    knn_g = knn_df[knn_df["gene"] == gene]
    knn_ku_vals = knn_g[knn_g["direction"] == "up"]["delta_pn_frac"].values
    knn_kd_vals = knn_g[knn_g["direction"] == "down"]["delta_pn_frac"].values
    knn_ku = knn_ku_vals.mean()
    knn_kd = knn_kd_vals.mean()

    lda_opp    = (lda_ku > 0) != (lda_kd > 0)
    knn_opp    = (knn_ku > 0) != (knn_kd > 0)
    lda_ct_agr = len(set(np.sign(lda_ku_vals))) == 1
    knn_ct_agr = len(set(np.sign(knn_ku_vals))) == 1
    lda_exp    = (lda_ku > 0) == ku_expected_positive
    knn_exp    = (knn_ku > 0) == ku_expected_positive

    reliability_rows.append({
        "gene":              gene,
        "expr(PN-sALS)":     round(expr_diff, 2),
        "LDA KU":            round(lda_ku, 4),
        "LDA KD":            round(lda_kd, 4),
        "KNN KU":            round(knn_ku, 5),
        "KNN KD":            round(knn_kd, 5),
        "LDA KU/KD opp":     lda_opp,
        "KNN KU/KD opp":     knn_opp,
        "LDA KU ct agree":   lda_ct_agr,
        "KNN KU ct agree":   knn_ct_agr,
        "LDA KU exp=obs":    lda_exp,
        "KNN KU exp=obs":    knn_exp,
    })

reliability_df = pd.DataFrame(reliability_rows)
reliability_df["lda_reliability"] = reliability_df[["LDA KU/KD opp", "LDA KU ct agree", "LDA KU exp=obs"]].sum(axis=1)
reliability_df["knn_reliability"] = reliability_df[["KNN KU/KD opp", "KNN KU ct agree", "KNN KU exp=obs"]].sum(axis=1)
reliability_df = reliability_df.sort_values(
    ["lda_reliability", "knn_reliability"], ascending=False
).reset_index(drop=True)

bool_cols = ["LDA KU/KD opp", "KNN KU/KD opp", "LDA KU ct agree", "KNN KU ct agree",
             "LDA KU exp=obs", "KNN KU exp=obs"]

def fmt(val):
    if isinstance(val, bool):
        return "✓" if val else "✗"
    return val

print(reliability_df.to_string(index=False, formatters={col: fmt for col in bool_cols}))

In [ ]:
# Save metric DataFrames and reliability table for Task 4
lda_df.to_csv("../data/lda_df.csv", index=False)
knn_df.to_csv("../data/knn_df.csv", index=False)
reliability_df.to_csv("../data/reliability_df.csv", index=False)
print("Saved lda_df.csv, knn_df.csv, reliability_df.csv")